# S0 · Mapa de offsets de λ por spaxel (checklist G1)

**Spec:** [`docs/plan_wavesol_stripes_2026-07-17.md`](../docs/plan_wavesol_stripes_2026-07-17.md)  |  **Bloque:** S · wavesol/stripes  |  **Run por defecto:** `ROXs12b_realigned`

Mide, spaxel a spaxel, el corrimiento espectral de las líneas de absorción de la primaria contra un espectro de referencia de campo (Xie+20 §4.2.2). Estructura alineada con slicers ⇒ diferencias de solución de λ por exposición/slice (*stripes*, Hashimoto+20). Es el **insumo formal de la decisión G1**.

| | |
|---|---|
| **Entrada** | `cube_telcorr.fits` (realineado) + ADP oficial de ESO (control independiente) |
| **Salida (QC/productos)** | `stages/stageS0_qc.json` (+ `stageS0_adp_qc.json`), `stages/stageS0_offset_map.fits`, `plots/s0_wavesol/` |
| **Consume aguas abajo** | **Decisión G1 (humana)** → Fase 2 (S2–S5) o cierre `fase2_descartable` |


## Qué hace S0 y cómo

Por cada spaxel del halo (selección por brillo, percentil 50) se normaliza el continuo por división de *running-median* (mata el continuo cromático del halo AO) en 4 ventanas de absorción estelar que **evitan** el láser AO, Hα (la primaria es emisora), y las bandas telúricas O₂/H₂O; se cross-correla contra el espectro de referencia del campo (`stripes._xcorr_shift_pixels`, subpíxel) y se toma la mediana de las ventanas usables. El resultado es un **mapa de offset** (Å) por spaxel.

**Corte S/N (lección del preliminar 2026-07-17):** cada spaxel lleva un error `err = σ_robusta(offsets_por_ventana)/√N`; el p95 del gate se calcula SOLO sobre spaxels con `err < max_err_ch` (0.08 ch ≈ 0.1 Å), porque el p95 crudo lo dominan spaxels débiles donde la xcorr falla (preliminar: p95 global 3.9 Å de puro ruido vs 0.18 Å en el núcleo r<1"). El perfil por columnas usa todos (su mediana ya es robusta).

La **métrica de estructura** colapsa el mapa a lo largo de la dirección de los stripes (perfil por columna) y compara su amplitud contra el ruido esperado; el perfil **transversal** es el control: stripes reales muestran estructura en el perfil de stripe, no en el transversal.


## Cómo ejecutar de forma independiente

Diagnóstico re-ejecutable sobre un cubo existente (no re-reduce nada); rutas fijas, sin `--run-id`:

```bash
conda activate MUSE               # kernel/env con astropy + musepipe
cd MUSE-accretion-pipeline                    # raíz del repo
python -m musepipe.qc.wavesol_map \
  --cube /mnt/2TB/MUSE_work/ROXs12b_realigned/cube_telcorr.fits \
  --qc-output runs/ROXs12b_realigned/stages/stageS0_qc.json \
  --map-output runs/ROXs12b_realigned/stages/stageS0_offset_map.fits \
  --plot-output runs/ROXs12b_realigned/plots/s0_wavesol/s0_realigned.png \
  --orientation vertical
python -m musepipe.qc.wavesol_map \
  --cube ../Data/ROX12b/20220829/ADP.2022-09-12T17_17_39.371.fits \
  --qc-output runs/ROXs12b_realigned/stages/stageS0_adp_qc.json \
  --map-output runs/ROXs12b_realigned/stages/stageS0_adp_offset_map.fits \
  --plot-output runs/ROXs12b_realigned/plots/s0_wavesol/s0_adp.png \
  --orientation vertical
```

Coste: full-res 330×338, normalización vectorizada; ~minutos por cubo.

La celda `RUN=True` de más abajo hace lo mismo desde el notebook.


In [ ]:
import os, sys
# Añade notebooks/ (para _nbcommon) y la RAÍZ del repo (para importar musepipe),
# funcione el cwd en notebooks/ o en la raíz del repo.
_here = os.getcwd()
if os.path.basename(_here) != 'notebooks' and os.path.isdir(os.path.join(_here, 'notebooks')):
    _here = os.path.join(_here, 'notebooks')
for _p in (_here, os.path.dirname(_here)):
    if _p not in sys.path:
        sys.path.insert(0, _p)
import _nbcommon as nb
_root = str(nb.project_root())
if _root not in sys.path:
    sys.path.insert(0, _root)   # asegura 'import musepipe'
RUN_ID = nb.resolve_run_id(None)
print('run  =', RUN_ID)
print('root =', _root)
print('dir  =', nb.run_dir(RUN_ID))


## Ejecutar o auditar


In [ ]:
RUN = False   # -> True para RE-EJECUTAR esta etapa (regenera su QC)

if RUN:
    cmd = 'python -m musepipe.qc.wavesol_map \\\n  --cube /mnt/2TB/MUSE_work/ROXs12b_realigned/cube_telcorr.fits \\\n  --qc-output runs/ROXs12b_realigned/stages/stageS0_qc.json \\\n  --map-output runs/ROXs12b_realigned/stages/stageS0_offset_map.fits \\\n  --plot-output runs/ROXs12b_realigned/plots/s0_wavesol/s0_realigned.png \\\n  --orientation vertical\npython -m musepipe.qc.wavesol_map \\\n  --cube ../Data/ROX12b/20220829/ADP.2022-09-12T17_17_39.371.fits \\\n  --qc-output runs/ROXs12b_realigned/stages/stageS0_adp_qc.json \\\n  --map-output runs/ROXs12b_realigned/stages/stageS0_adp_offset_map.fits \\\n  --plot-output runs/ROXs12b_realigned/plots/s0_wavesol/s0_adp.png \\\n  --orientation vertical'.replace('$RUN', RUN_ID)
    print('ejecutando:', cmd)
    import subprocess
    subprocess.run(cmd, shell=True, cwd=str(nb.project_root()), check=True)
else:
    print('Modo auditoría (RUN=False): se carga el QC existente abajo.')


## QC / resultados


In [ ]:
qc = nb.load_qc('stages/stageS0_qc.json', RUN_ID)
nb.show(qc, keys=['gate_g1.recommendation', 'gate_g1.decision', 'metrics.p95_abs_offset_A', 'metrics.structure_significance', 'metrics.transverse_significance', 'metrics.n_selected_low_err', 'channel_step_A', 'runtime_s'], title='S0')


## Evidencia: realineado vs ADP (control)

Los dos cubos deben coincidir: el mapa de offset es un diagnóstico del **instrumento/reducción**, no del cubo concreto. El preliminar 2×2 (2026-07-17) dio amplitud de columna 72 mÅ (realineado) / 67 mÅ (ADP), a ~1× ruido, sin estructura alineada con slicers; el núcleo r<1" med|off| = 64 / 62 mÅ ≈ el M1 global (+74 mÅ).

**Full-res reproduce la conclusión clave** (sin estructura de slicer: `stripe_sig` ≈ control transversal) y la extiende: al medir TODOS los spaxels de bajo error (no solo el núcleo) el p95 sube a ~0.32 Å — un scatter de λ por spaxel, consistente entre ventanas pero **espacialmente desestructurado**. La celda imprime ambos cubos; deben coincidir.


In [ ]:
# p95 se mide sobre el subconjunto de bajo error (err<max_err_ch); la
# estructura de slicer se juzga con stripe_sig vs el control transversal.
rows = []
for label, rel in [('realineado', 'stages/stageS0_qc.json'),
                   ('ADP',        'stages/stageS0_adp_qc.json')]:
    try:
        q = nb.load_qc(rel, RUN_ID)
    except FileNotFoundError:
        print(f'[{label}] QC aún no existe: {rel}'); continue
    m = q['metrics']
    rows.append((label, m['p95_abs_offset_A'], m['structure_significance'],
                 m['transverse_significance'], m['n_selected_low_err'],
                 m['n_spaxels_measured'], q['gate_g1']['recommendation']))
hdr = ('cubo', 'p95|off|[A]', 'stripe_sig', 'transv_sig', 'n_low_err',
       'n_meas', 'recomendación')
print('{:>10} {:>12} {:>11} {:>11} {:>10} {:>8}  {}'.format(*hdr))
for r in rows:
    print('{:>10} {:>12.4f} {:>11.2f} {:>11.2f} {:>10d} {:>8d}  {}'.format(*r))
print('\np95 sobre spaxels de bajo error; stripe_sig<=transv_sig => sin estructura de slicer.')


## Checklist G1 — umbrales y argumentos

**Umbrales del plan (recomendación automática, decisión humana):**

| Criterio | Umbral | Dispara Fase 2 si |
|---|---|---|
| p95 \|offset\| (spaxels `err<0.1 Å`) | 0.1 Å | **>** 0.1 Å |
| Estructura alineada con slicers | 3× ruido | **>** 3× **y** > 2× el control transversal |

**Argumentos del caso `fase2_descartable` (documentados aunque la decisión sea seguir):**

1. La métrica espacial de S0 **no ve un offset común a todas las exposiciones** (deriva temporal uniforme): ese modo no aparece como estructura espacial, solo **ensancharía la LSF combinada**.
2. Pero A4·M2 midió **LSF = 2.383 Å**, MÁS ESTRECHA que el nominal → acota ese *smearing* a nivel pequeño (si hubiera deriva grande entre exposiciones, la LSF combinada saldría ensanchada, no estrecha).
3. El **M1 global (+0.074 Å)** ya corrige el zero-point de λ.

Los tres juntos son el caso para **no** entrar a la Fase 2 (re-reducción por exposición). La decisión final es **humana** (gate G1).


## Mapas S0 (offset, error, perfil de columna, histograma)

Renderizados **directamente del FITS** `stageS0_offset_map.fits` (ext `OFFSET_A`, `ERR_A`, `OFFSET_CH`, `ERR_CH`), para **ambos cubos** (realineado y ADP-control) — no depende del PNG pre-generado. Cada fila es un cubo con 4 paneles: mapa de offset (Å), mapa de error (Å), perfil colapsado por columna (dirección de los stripes) y el histograma de offsets sobre los spaxels de bajo error.


In [ ]:
import numpy as np, matplotlib.pyplot as plt
from astropy.io import fits
import musepipe.qc.wavesol_map as wsm
rd = nb.run_dir(RUN_ID)
CUBES = [('realineado', 'stageS0_qc.json', 'stageS0_offset_map.fits'),
         ('ADP (control)', 'stageS0_adp_qc.json', 'stageS0_adp_offset_map.fits')]
for label, qcf, mapf in CUBES:
    mp = rd / 'stages' / mapf
    if not mp.exists():
        print(f'[{label}] falta {mp} — corre la etapa (arriba).'); continue
    q = nb.load_qc(f'stages/{qcf}', RUN_ID)
    step = q['channel_step_A']; thr = q['gate_g1']['thresholds']['max_err_ch']
    with fits.open(mp) as h:
        off_A = h['OFFSET_A'].data.astype(float); err_A = h['ERR_A'].data.astype(float)
        off_ch = h['OFFSET_CH'].data.astype(float); err_ch = h['ERR_CH'].data.astype(float)
    low = np.isfinite(off_A) & np.isfinite(err_ch) & (err_ch < thr)
    prof = wsm.stripe_profile(off_ch, 'vertical')
    fig, ax = plt.subplots(1, 4, figsize=(17, 3.6))
    fig.suptitle(f'S0 · {label}  (bajo error err<{thr} ch: {int(low.sum())} spaxels)', fontsize=11)
    vlim = np.nanpercentile(np.abs(off_A[low]), 95) if low.any() else 0.3
    im0 = ax[0].imshow(off_A, origin='lower', cmap='RdBu_r', vmin=-vlim, vmax=vlim)
    ax[0].set_title('offset (Å)'); plt.colorbar(im0, ax=ax[0], fraction=0.046)
    im1 = ax[1].imshow(err_A, origin='lower', cmap='viridis', vmax=np.nanpercentile(err_A, 95))
    ax[1].set_title('error (Å)'); plt.colorbar(im1, ax=ax[1], fraction=0.046)
    p_A = np.asarray(prof['profile'], float) * step
    ax[2].plot(np.arange(p_A.size), p_A, lw=0.9, color='tab:blue')
    ax[2].axhline(0, color='0.6', lw=0.6); ax[2].set_xlabel('columna'); ax[2].set_ylabel('offset mediano (Å)')
    ax[2].set_title('perfil por columna (∥ stripes)')
    ax[3].hist(off_A[low], bins=60, color='tab:blue', alpha=0.8)
    ax[3].axvline(0, color='k', lw=0.8); ax[3].axvline(np.median(off_A[low]), color='tab:red', ls='--', lw=1, label=f'mediana {np.median(off_A[low])*1e3:+.0f} mÅ')
    ax[3].set_xlabel('offset (Å)'); ax[3].set_title('hist (bajo error)'); ax[3].legend(fontsize=8)
    fig.tight_layout(); plt.show()


## Paso extra — crop 100×100 en la estrella (alta S/N)

**Motivación:** el gate global está lastrado por spaxels débiles del borde (el p95 crudo y el `median|offset|` los dominan). Si hubiera un offset **coherente pequeño escondido en el ruido**, la forma de sacarlo a la luz es medirlo donde la S/N es máxima: el halo de la estrella central. Este paso recorta un **100×100 centrado en la estrella** (centroide de los spaxels de menor error, sin cargar el cubo) y recalcula las métricas de S0 ahí, más el **offset medio con signo ± error** — el test directo de un offset coherente oculto (que el `median|·|` no ve porque no distingue signo).

Se compara *full-frame* vs *crop* para ambos cubos; la conclusión robusta es que en el crop (i) el `median|offset|` cae al quitar el ruido de bordes, y (ii) `stripe_sig` sigue ≤ el control transversal → **ningún stripe alineado con slicers aparece al subir la S/N**; lo que queda es un zero-point casi uniforme (unas decenas de mÅ, ≲0.04 canal), coherente con la interpretación temporal del G1 y con M1 (+74 mÅ).


In [ ]:
import numpy as np, matplotlib.pyplot as plt
from astropy.io import fits
import musepipe.qc.wavesol_map as wsm
rd = nb.run_dir(RUN_ID)
HALF = 50   # crop 100x100
CUBES = [('realineado', 'stageS0_qc.json', 'stageS0_offset_map.fits'),
         ('ADP (control)', 'stageS0_adp_qc.json', 'stageS0_adp_offset_map.fits')]
print('{:>16} {:>6} {:>10} {:>9} {:>9} {:>7} {:>7} {:>6}'.format(
      'cubo/región', 'n_lowE', 'med|off|mÅ', 'p95 Å', 'mean mÅ', 'σ_mean', 'stripe', 'transv'))
def stats(oc, ec, oa, step, thr):
    m = wsm.structure_metrics(oc, step, err_map_ch=ec, max_err_ch=thr)
    low = np.isfinite(oa) & np.isfinite(ec) & (ec < thr); v = oa[low]
    mean = float(np.mean(v)); se = float(np.std(v) / np.sqrt(max(v.size, 1)))
    return m, mean, se
for label, qcf, mapf in CUBES:
    mp = rd / 'stages' / mapf
    if not mp.exists():
        print(f'[{label}] falta {mp}'); continue
    q = nb.load_qc(f'stages/{qcf}', RUN_ID)
    step = q['channel_step_A']; thr = q['gate_g1']['thresholds']['max_err_ch']
    with fits.open(mp) as h:
        oc = h['OFFSET_CH'].data.astype(float); oa = h['OFFSET_A'].data.astype(float)
        ec = h['ERR_CH'].data.astype(float)
    ny, nx = oc.shape
    low = np.isfinite(oc) & np.isfinite(ec) & (ec < thr)
    yy, xx = np.mgrid[0:ny, 0:nx]; wt = np.where(low, 1.0/np.clip(ec, 1e-3, None)**2, 0.0)
    cy = int(round(np.sum(yy*wt)/np.sum(wt))); cx = int(round(np.sum(xx*wt)/np.sum(wt)))
    y0, y1 = max(0, cy-HALF), min(ny, cy+HALF); x0, x1 = max(0, cx-HALF), min(nx, cx+HALF)
    mF, meanF, seF = stats(oc, ec, oa, step, thr)
    sl = (slice(y0, y1), slice(x0, x1))
    mC, meanC, seC = stats(oc[sl], ec[sl], oa[sl], step, thr)
    for tag, mm, mean, se in [('FULL', mF, meanF, seF), (f'CROP@({cy},{cx})', mC, meanC, seC)]:
        print('{:>16} {:>6d} {:>10.0f} {:>9.4f} {:>+9.1f} {:>7.1f} {:>7.2f} {:>6.2f}'.format(
              f'{label[:9]} {tag}', mm['n_selected_low_err'], mm['median_abs_offset_ch']*step*1e3,
              mm['p95_abs_offset_A'], mean*1e3, se*1e3, mm['structure_significance'], mm['transverse_significance']))
    # render crop offset map + hist (this cube)
    lowc = np.isfinite(oa[sl]) & np.isfinite(ec[sl]) & (ec[sl] < thr)
    vlim = np.nanpercentile(np.abs(oa[sl][lowc]), 95) if lowc.any() else 0.3
    fig, ax = plt.subplots(1, 2, figsize=(9.5, 3.8))
    im = ax[0].imshow(oa[sl], origin='lower', cmap='RdBu_r', vmin=-vlim, vmax=vlim)
    ax[0].set_title(f'{label} · crop offset (Å)'); plt.colorbar(im, ax=ax[0], fraction=0.046)
    ax[1].hist(oa[sl][lowc], bins=45, color='tab:green', alpha=0.8)
    ax[1].axvline(0, color='k', lw=0.8)
    ax[1].axvline(meanC, color='tab:red', ls='--', lw=1.2, label=f'media {meanC*1e3:+.1f}±{seC*1e3:.1f} mÅ ({abs(meanC/seC):.0f}σ)')
    ax[1].set_xlabel('offset (Å)'); ax[1].set_title('crop hist (bajo error)'); ax[1].legend(fontsize=8)
    fig.tight_layout(); plt.show()
print('\nLectura: en el crop el median|offset| cae (se va el ruido de bordes) y el offset medio')
print('con signo se resuelve a unas decenas de mÅ (≲0.04 canal); stripe_sig <= transversal =>')
print('NO es un stripe de slicer al subir la S/N, sino un zero-point casi uniforme (temporal, M1).')


## Decisiones y notas
- S0a/S0b: núcleo target-agnostic (`musepipe/qc/wavesol_map.py`) + CLI, normalización vectorizada (gate de equivalencia <1e-9) y corte S/N por spaxel; 14 tests verdes. · [`docs/plan_wavesol_stripes_pasos_agente.md`](../docs/plan_wavesol_stripes_pasos_agente.md)
- El offset map es diagnóstico del instrumento/reducción: realineado ≈ ADP (control cruzado). · [`docs/plan_wavesol_stripes_2026-07-17.md`](../docs/plan_wavesol_stripes_2026-07-17.md)
- **Gate G1 (humano):** con este producto se decide entrar o no a la Fase 2 (S2–S5, re-reducción por exposición). La recomendación automática se imprime en Checks. · [`docs/plan_wavesol_stripes_2026-07-17.md`](../docs/plan_wavesol_stripes_2026-07-17.md)
- **DECIDIDO 2026-07-17:** cerrar como sistemático acotado (interpretación temporal); Fase 2 NO disparada; confirmación diferida (S0 por exposición cuando existan los 7 cubos). Anula la recomendación automática. · [`docs/decision_g1_wavesol_2026-07-17.md`](../docs/decision_g1_wavesol_2026-07-17.md)


## Checks


In [ ]:
q = nb.load_qc('stages/stageS0_qc.json', RUN_ID)
m, g = q['metrics'], q['gate_g1']
th = g['thresholds']
p95 = m['p95_abs_offset_A']; sig = m['structure_significance']; sigt = m['transverse_significance']
c1 = p95 <= th['p95_threshold_A']
aligned = (sig > th['significance_threshold']) and (sigt != sigt or sig > 2.0 * sigt)
print('CHECKLIST G1 (realineado, full-res):')
print(f"  p95|off| = {p95:.4f} A  (umbral {th['p95_threshold_A']} A)   -> {'OK pequeño' if c1 else 'GRANDE'}")
print(f"  estructura stripe = {sig:.2f}x ruido   (control transversal {sigt:.2f}x, umbral {th['significance_threshold']}x)")
print(f"  {'sin' if not aligned else 'CON'} estructura alineada con slicers dominante")
print(f"  n_spaxels bajo corte err<{th['max_err_ch']} ch = {m['n_selected_low_err']} / {m['n_spaxels_measured']} medidos")
print(f"\n  recomendación automática: {g['recommendation']}")
for r in g['reasons']:
    print('   -', r)
hd = q.get('g1_human_decision')
if hd:
    print(f"\n  DECISIÓN G1 (humana, {hd['date']}): {hd['decision']} "
          f"[interpretación: {hd['interpretation']}; anula {hd['recommendation_overridden']}]")
    print(f"    fase2_triggered = {hd['phase2_triggered']}; confirmación diferida: {hd['deferred_confirmation'][:80]}...")
    print(f"    doc: {hd['doc']}")
else:
    print(f"  decisión: {g['decision']} (aún no registrada en este QC)")


## Conclusión (registrada, 2026-07-17) — G1 DECIDIDO

**S0 (full-res, 330×338):** sin estructura de slicer (`stripe_sig` ≤ control transversal) y **realineado ≈ ADP**, pero p95\|off\| ≈ 0.32 Å (>0.1 Å), scatter por spaxel creciente con el radio.

**Clave (por qué el cubo combinado es ciego a los stripes):** las 7 exposiciones son de una noche, dithers ≈0, pero el **campo rota 5.9° (PA)** entre ellas (derotador `ABSROT` barre 19.3°). Un stripe fijo en el slicer se promedia acimutalmente al combinar ⇒ aparece como scatter desestructurado creciente con el radio (~7.5 spaxels en el compañero a 1.80″). **S0 sobre el combinado no puede confirmar ni descartar stripes**; el 'sin estructura' es esperable en cualquier caso.

➡️ **DECISIÓN G1 (humana, 2026-07-17): cerrar como sistemático acotado, interpretación TEMPORAL** (deriva de zero-point por exposición, acotada por A4·M2 LSF=2.383 Å). **Fase 2 NO disparada.** Anula la recomendación automática (`fase2_justificada`, que salía solo por el p95). **Salvedad:** M2 acota el modo temporal uniforme, no los stripes rotados; defendible para la no-detección de Hα / límites (E1/E3), más débil para líneas finas en G2/G3.

**Confirmación (HECHA 2026-07-18):** se regeneraron los 7 cubos por exposición (S2) y se corrió **S0 por exposición** (0/7 con estructura de slicer) **+ S3** (deriva temporal ~0.04 Å std). Ambas ramas cerradas ⇒ **GATE G1 CERRADO** (`gate_g1.decision=closed`). Detalle en `docs/decision_g1_wavesol_2026-07-17.md`.

**Corroboración con el crop 100×100 en la estrella (paso extra):** al restringir a la región de máxima S/N, el `median|offset|` cae de ~0.8 Å a ~0.11 Å (era ruido de bordes) y el offset medio con signo se resuelve a **unas decenas de mÅ** (realineado ≈ −48 mÅ, ADP ≈ −11 mÅ; ≲0.04 canal), pero **`stripe_sig` sigue ≤ el control transversal** en ambos cubos → **ningún stripe de slicer emerge al subir la S/N**. Lo que queda es un zero-point casi uniforme, del orden de la deriva temporal S3 (~0.04 Å) y dentro de lo que M1 (+74 mÅ) corrige — refuerza el cierre G1 al mejor S/N disponible.
